# End-to-End ML Training Notebook (Customer Churn Risk)

This notebook upgrades the project into 4 serious modeling labs under one unified workflow:
1. LazyPredict Discovery Lab
2. Manual Engineering Lab
3. FLAML Optimization Lab
4. PyCaret Experiment Lab

All tracks feed one unified leaderboard and a final deployment decision.


## 1) Business Problem and Success Criteria

### Problem
Predict customer churn risk early enough to trigger retention action.

### Success Criteria
- Optimize **expected business cost per 1000 predictions** (primary metric).
- Maintain strong predictive quality via PR-AUC / ROC-AUC.
- Respect operational constraints (latency + model size + retrain effort).
- Produce deployment-ready artifact + monitoring-ready outputs.


In [ ]:
from __future__ import annotations

import json
import shutil
from pathlib import Path

import numpy as np
import pandas as pd

from src.data_pipeline import TARGET_COLUMN, load_dataset, split_dataset, build_preprocessor
from src.infer import load_runtime
from src.train import (
    EXPORT_COLUMNS,
    enrich_with_ranking,
    run_baseline_track,
    run_lazypredict_discovery_lab,
    run_manual_engineering_lab,
    run_flaml_optimization_lab,
    run_pycaret_experiment_lab,
    run_seed_stability_check,
    select_winner,
)

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

DATA_PATH = Path("data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv")
ARTIFACTS_DIR = Path("artifacts")
(ARTIFACTS_DIR / "reports").mkdir(parents=True, exist_ok=True)
(ARTIFACTS_DIR / "models").mkdir(parents=True, exist_ok=True)
(ARTIFACTS_DIR / "model_registry").mkdir(parents=True, exist_ok=True)
(ARTIFACTS_DIR / "monitoring").mkdir(parents=True, exist_ok=True)

SEED = 42
FP_COST = 65.0
FN_COST = 320.0
FLAML_TIME_BUDGET = 90
MANUAL_TOP_N = 3
LAZY_TOP_K = 12


## 2) Dataset Access and Data Dictionary


In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found: {DATA_PATH}\n"
        "Run first: uv run --with kaggle kaggle datasets download -d blastchar/telco-customer-churn -p data/raw --unzip"
    )

df = load_dataset(DATA_PATH)
print("Dataset shape:", df.shape)
df.head(5)


In [ ]:
data_dict = pd.DataFrame(
    {
        "feature": df.columns,
        "dtype": [str(df[col].dtype) for col in df.columns],
        "null_count": [int(df[col].isna().sum()) for col in df.columns],
        "null_pct": [float(df[col].isna().mean()) for col in df.columns],
        "n_unique": [int(df[col].nunique(dropna=True)) for col in df.columns],
    }
).sort_values(["null_count", "n_unique"], ascending=[False, False])

data_dict


## 3) Data Cleaning and Leakage Checks

Cleaning is handled in `src.data_pipeline.load_dataset()`:
- target normalization (`Yes/No -> 1/0`)
- numeric coercion (`TotalCharges`)
- duplicate removal


In [ ]:
duplicate_rows = int(df.duplicated().sum())
label_balance = df[TARGET_COLUMN].value_counts(normalize=True).rename("rate")

leakage_candidates = [
    col
    for col in df.columns
    if any(token in col.lower() for token in ["churn", "cancel", "target", "label"])
    and col != TARGET_COLUMN
]

print("Duplicate rows:", duplicate_rows)
print("Leakage candidate columns:", leakage_candidates if leakage_candidates else "None detected by heuristic")
print("Label balance:")
display(label_balance)


## 4) Feature Engineering

- Categorical: mode imputation + one-hot encoding
- Numeric: median imputation + standard scaling
- Customer identifier dropped before modeling


In [ ]:
splits = split_dataset(df, random_state=SEED)

X_train, y_train = splits.X_train, splits.y_train
X_valid, y_valid = splits.X_valid, splits.y_valid
X_test, y_test = splits.X_test, splits.y_test

preprocessor = build_preprocessor(X_train)
X_train_matrix = preprocessor.fit_transform(X_train)

if hasattr(X_train_matrix, "shape"):
    print("Transformed train matrix shape:", X_train_matrix.shape)

numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = [c for c in X_train.columns if c not in numeric_cols]
print("Numeric features:", len(numeric_cols))
print("Categorical features:", len(categorical_cols))


## 5) Validation Strategy

We use a strict split discipline:
- train: model fitting / CV
- validation: threshold tuning and model-family selection
- holdout test: final comparison metrics


In [ ]:
split_summary = pd.DataFrame(
    {
        "split": ["train", "valid", "test"],
        "n_rows": [len(X_train), len(X_valid), len(X_test)],
        "churn_rate": [y_train.mean(), y_valid.mean(), y_test.mean()],
    }
)
split_summary


In [ ]:
baseline_records, baseline_artifacts = run_baseline_track(
    X_train=X_train,
    y_train=y_train,
    X_valid=X_valid,
    y_valid=y_valid,
    X_test=X_test,
    y_test=y_test,
    fp_cost=FP_COST,
    fn_cost=FN_COST,
    artifacts_dir=ARTIFACTS_DIR,
    seed=SEED,
    log_mlflow=False,
    persist_artifacts=True,
)

pd.DataFrame(baseline_records)


## 6) LazyPredict Discovery Lab

Purpose: rapid model-family discovery **after** proper feature engineering and validation setup.

Outputs:
- ranked benchmark table
- eligibility filtering
- top 3 eligible families for manual implementation


In [ ]:
lazy_records, lazy_ranked, lazy_top3 = run_lazypredict_discovery_lab(
    X_train=X_train,
    y_train=y_train,
    X_valid=X_valid,
    y_valid=y_valid,
    fp_cost=FP_COST,
    fn_cost=FN_COST,
    top_k=LAZY_TOP_K,
    manual_top_n=MANUAL_TOP_N,
    artifacts_dir=ARTIFACTS_DIR,
    log_mlflow=False,
)

lazy_ranked.head(15)


In [ ]:
print("LazyPredict benchmark rows:", len(lazy_ranked))
print("Eligible manual families discovered:", lazy_top3["manual_family"].dropna().tolist())
lazy_top3


**LazyPredict conclusion:** the manual track must only use the top eligible families from this lab.


## 7) Selection of Top 3 Eligible Models


In [ ]:
selected_families = lazy_top3["manual_family"].dropna().astype(str).tolist()[:MANUAL_TOP_N]

if len(selected_families) < MANUAL_TOP_N:
    raise RuntimeError(f"Expected {MANUAL_TOP_N} manual families, got {selected_families}")

selection_table = lazy_top3[["lazy_rank", "lazy_model_name", "manual_family", "eligibility_reason"]].copy()
selection_table


## 8) Manual Engineering Lab

Manual implementation is constrained to the selected families from Section 7.

Includes:
- explicit preprocessing pipeline
- CV + holdout metrics
- threshold optimization by business cost
- calibration where relevant
- error analysis artifacts


In [ ]:
manual_records, manual_artifacts, manual_diag = run_manual_engineering_lab(
    X_train=X_train,
    y_train=y_train,
    X_valid=X_valid,
    y_valid=y_valid,
    X_test=X_test,
    y_test=y_test,
    selected_families=selected_families,
    lazy_top3=lazy_top3,
    fp_cost=FP_COST,
    fn_cost=FN_COST,
    artifacts_dir=ARTIFACTS_DIR,
    seed=SEED,
    log_mlflow=False,
    persist_artifacts=True,
)

manual_df = pd.DataFrame(manual_records)
manual_df


In [ ]:
manual_diag


**Manual lab conclusion:** compare cost efficiency and calibration quality before picking a manual champion.


## 9) FLAML Optimization Lab

This is a full optimization track with:
- explicit `time_budget`
- business-cost-centric search objective (fallback to ROC-AUC only if needed)
- searchable summary of estimators and tuned configurations


In [ ]:
flaml_records, flaml_artifacts, flaml_search_summary, flaml_best_config = run_flaml_optimization_lab(
    X_train=X_train,
    y_train=y_train,
    X_valid=X_valid,
    y_valid=y_valid,
    X_test=X_test,
    y_test=y_test,
    fp_cost=FP_COST,
    fn_cost=FN_COST,
    time_budget=FLAML_TIME_BUDGET,
    artifacts_dir=ARTIFACTS_DIR,
    seed=SEED,
    log_mlflow=False,
    persist_artifacts=True,
)

pd.DataFrame(flaml_records)


In [ ]:
print("FLAML best config:")
print(json.dumps(flaml_best_config, indent=2))

flaml_search_summary.head(20)


**FLAML conclusion:** verify whether AutoML found a model with better cost-efficiency or stronger ops profile than manual engineering.


## 10) PyCaret Experiment Lab

This track runs a structured PyCaret flow:
- `setup()`
- `compare_models()`
- `tune_model()`
- `calibrate_model()`
- `finalize_model()` + `save_model()`


In [ ]:
pycaret_records, pycaret_artifacts, pycaret_tables = run_pycaret_experiment_lab(
    X_train=X_train,
    y_train=y_train,
    X_valid=X_valid,
    y_valid=y_valid,
    X_test=X_test,
    y_test=y_test,
    fp_cost=FP_COST,
    fn_cost=FN_COST,
    artifacts_dir=ARTIFACTS_DIR,
    seed=SEED,
    log_mlflow=False,
    persist_artifacts=True,
)

pd.DataFrame(pycaret_records)


In [ ]:
if "compare" in pycaret_tables and not pycaret_tables["compare"].empty:
    display(pycaret_tables["compare"].head(10))

if "tune" in pycaret_tables and not pycaret_tables["tune"].empty:
    display(pycaret_tables["tune"].head(10))

if "calibration" in pycaret_tables and not pycaret_tables["calibration"].empty:
    display(pycaret_tables["calibration"].head(10))


**PyCaret conclusion:** keep the finalized model only if it is competitive on business cost and operational constraints.


## 11) Unified Leaderboard and Final Model Ranking

We combine:
- baseline
- top LazyPredict benchmark rows
- manual top-3 implementations
- best FLAML result
- best PyCaret finalized result


In [ ]:
all_records = baseline_records + lazy_records + manual_records + flaml_records + pycaret_records
leaderboard = enrich_with_ranking(pd.DataFrame(all_records))

for col in EXPORT_COLUMNS:
    if col not in leaderboard.columns:
        leaderboard[col] = np.nan

leaderboard_export = leaderboard[EXPORT_COLUMNS].copy()
leaderboard_path = ARTIFACTS_DIR / "leaderboard_e2e.csv"
leaderboard_export.to_csv(leaderboard_path, index=False)

artifact_map = {}
artifact_map.update(baseline_artifacts)
artifact_map.update(manual_artifacts)
artifact_map.update(flaml_artifacts)
artifact_map.update(pycaret_artifacts)

winner, scoring_table = select_winner(
    leaderboard=leaderboard,
    deployable_ids=set(artifact_map.keys()),
    max_p95_latency_ms=40.0,
    min_secondary_metric=0.45,
    max_calibration_metric=0.30,
)

print("Leaderboard saved:", leaderboard_path)
print("Winner:", winner["library_source"], winner["model_name"], "rank_score=", round(float(winner["rank_score"]), 3))

leaderboard.head(10)


In [ ]:
deployable_ranked = leaderboard[leaderboard["_candidate_id"].isin(artifact_map.keys())].sort_values("rank_score", ascending=False)
final_top3 = deployable_ranked.head(3)

stability_df = run_seed_stability_check(
    top_candidates=final_top3,
    seeds=[11, 42, 87],
    X_train=X_train,
    y_train=y_train,
    X_valid=X_valid,
    y_valid=y_valid,
    X_test=X_test,
    y_test=y_test,
    fp_cost=FP_COST,
    fn_cost=FN_COST,
    flaml_time_budget=FLAML_TIME_BUDGET,
    artifacts_dir=ARTIFACTS_DIR,
)

stability_path = ARTIFACTS_DIR / "reports" / "top3_seed_stability.csv"
stability_df.to_csv(stability_path, index=False)
print("Seed stability report:", stability_path)

stability_df.head(30)


## 12) Business Recommendation


In [ ]:
deployable_ranked = leaderboard[leaderboard["_candidate_id"].isin(artifact_map.keys())].sort_values("rank_score", ascending=False)
winner_row = deployable_ranked.iloc[0]
runner_up_row = deployable_ranked.iloc[1] if len(deployable_ranked) > 1 else None

print("Recommended production winner:")
print(
    f"- {winner_row['library_source']}::{winner_row['model_name']} | "
    f"cost/1000={winner_row['holdout_primary_metric']:.2f} | "
    f"PR-AUC={winner_row['holdout_secondary_metric']:.4f} | "
    f"latency(ms)={winner_row['infer_latency_ms']:.4f}"
)

if runner_up_row is not None:
    print("\nSecond-best safer option when stability or explainability is prioritized:")
    print(
        f"- {runner_up_row['library_source']}::{runner_up_row['model_name']} | "
        f"cost/1000={runner_up_row['holdout_primary_metric']:.2f} | "
        f"PR-AUC={runner_up_row['holdout_secondary_metric']:.4f} | "
        f"latency(ms)={runner_up_row['infer_latency_ms']:.4f}"
    )


## 13) Inference / Deployment Path


In [ ]:
selected_artifact_path = artifact_map[winner["_candidate_id"]]
final_model_path = ARTIFACTS_DIR / "model_registry" / "final_model.joblib"
shutil.copy2(selected_artifact_path, final_model_path)

runtime = load_runtime(final_model_path)
sample_records = X_test.head(3).to_dict(orient="records")
sample_predictions = runtime.predict_records(sample_records)

model_selection_report = {
    "selected_candidate_id": winner["_candidate_id"],
    "library_source": winner["library_source"],
    "model_name": winner["model_name"],
    "rank_score": float(winner["rank_score"]),
    "holdout_primary_metric": float(winner["holdout_primary_metric"]),
    "holdout_secondary_metric": float(winner["holdout_secondary_metric"]),
    "holdout_tertiary_metric": float(winner["holdout_tertiary_metric"])
    if pd.notna(winner["holdout_tertiary_metric"])
    else None,
}

report_path = ARTIFACTS_DIR / "model_selection_report.json"
report_path.write_text(json.dumps(model_selection_report, indent=2), encoding="utf-8")

print("Final model copied to:", final_model_path)
print("Model selection report:", report_path)
pd.DataFrame(sample_predictions)


## 14) Monitoring / Drift / Retraining Plan


In [ ]:
from src.evaluate import build_drift_table, plot_feature_distributions

recent_batch = X_test.sample(min(len(X_test), 250), random_state=SEED).copy()
if "MonthlyCharges" in recent_batch.columns:
    recent_batch["MonthlyCharges"] = recent_batch["MonthlyCharges"].astype(float) * 1.10

recent_batch_path = Path("data/recent/recent_batch.csv")
recent_batch_path.parent.mkdir(parents=True, exist_ok=True)
recent_batch.to_csv(recent_batch_path, index=False)

batch_predictions = pd.DataFrame(runtime.predict_records(recent_batch.to_dict(orient="records")))
drift_df = build_drift_table(X_train, recent_batch)

monitor_dir = ARTIFACTS_DIR / "monitoring"
monitor_dir.mkdir(parents=True, exist_ok=True)

drift_path = monitor_dir / "drift_indicators.csv"
score_path = monitor_dir / "recent_batch_scoring.csv"
plot_path = monitor_dir / "feature_distribution_comparison.png"

batch_predictions.to_csv(score_path, index=False)
drift_df.to_csv(drift_path, index=False)
plot_feature_distributions(X_train, recent_batch, plot_path)

high_drift_ratio = (drift_df["drift_level"] == "high").mean() if not drift_df.empty else 0.0

monitor_report = (
    "# Monitoring Snapshot\n\n"
    f"- recent batch size: {len(recent_batch)}\n"
    f"- mean churn probability: {batch_predictions['churn_probability'].mean():.4f}\n"
    f"- high-risk prediction rate: {(batch_predictions['predicted_label'] == 1).mean():.4f}\n"
    f"- high drift ratio: {high_drift_ratio:.2%}\n\n"
    "Retraining triggers:\n"
    "1. >25% monitored features with drift score > 0.20.\n"
    "2. business cost/1000 worsens by >=20% vs holdout baseline once labels arrive.\n"
    "3. scheduled 30-day retrain cadence with sufficient new labels.\n"
)

report_md_path = monitor_dir / "batch_scoring_report.md"
report_md_path.write_text(monitor_report, encoding="utf-8")

print("Monitoring report:", report_md_path)
print("Drift CSV:", drift_path)
print("Scoring CSV:", score_path)
print("Distribution plot:", plot_path)

drift_df.head(10)


## 15) Limitations and Next Steps

### Current limitations
- Runtime can be heavy because 4 labs are fully executed in one notebook.
- LazyPredict outputs are used for family discovery, not direct deployment artifacts.
- Seed-stability checks can be expensive for FLAML/PyCaret candidates.

### Practical next steps
1. Add SHAP-based explanation package for model governance.
2. Add CI checks that validate leaderboard schema and artifact presence.
3. Add online latency/load tests beyond local p95 single-row estimates.
4. Add labeled recent-batch backtesting for production drift-to-quality linkage.
